# Step 1 — Get raw video metadata from YouTube

Pipeline stage 1 of the DEI project: for every **CEO / year** pair in `data/ceos.csv`,
search YouTube for `"[CEO] interview"` restricted to that year and store **all** the
metadata of **every** video the API returns.

**Output:** `data/output/videos_metadata/all_videos.json` — one record per CEO/year pair:

```json
{
  "year": 2026, "company": "Amazon", "rank": 1, "CEO": "Andy Jassy",
  "query": "Andy Jassy interview",
  "n_videos_found": 50,
  "videos": [ { ...full YouTube metadata... } ]
}
```

- `n_videos_found` is an **integer** (possibly `0`) when the CEO is known.
- `n_videos_found` is the string **`"NA"`** when the CEO name is missing from `ceos.csv` —
  no search was possible, which is different from "searched and found nothing".

**Resumable:** the collection cell can be re-run as many times as needed. Pairs that
already have a result are skipped, never overwritten. Progress is flushed to disk after
*every* CEO, so an interruption (or exhausted quota) never loses work.

**Size:** roughly 5 KB per video → expect the finished `all_videos.json` to land around
**0.5–0.7 GB**. That is well past GitHub's 100 MB per-file limit, so add
`data/output/` to `.gitignore` (or track it with Git LFS) before committing.

**Quota:** each pair costs 101 units (`search.list` = 100, `videos.list` = 1). A YouTube
API key gets 10,000 units/day → ~99 pairs per key per day. Keys rotate automatically when
one runs out; when all are exhausted the run stops cleanly and you pick it up tomorrow.

In [ ]:
# !pip install google-api-python-client isodate python-dotenv pandas

## 1. Configuration

In [6]:
import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# ── Paths ────────────────────────────────────────────────────────────────────
ENV_FILE   = Path(".env")
CEOS_CSV   = Path("data/ceos.csv")
OUTPUT_DIR = Path("data/output/videos_metadata")
OUTPUT_JSON  = OUTPUT_DIR / "all_videos.json"   # the deliverable
OUTPUT_JSONL = OUTPUT_DIR / "all_videos.jsonl"  # append-only checkpoint (crash safety)
KEY_PROJECTS_CACHE = OUTPUT_DIR / "key_projects.json"  # which Cloud project each API key belongs to

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Search parameters ────────────────────────────────────────────────────────
QUERY_TEMPLATE = "{ceo} interview"
MAX_RESULTS    = 50    # YouTube search.list hard cap per request

# ── Re-run behaviour ─────────────────────────────────────────────────────────
# Pairs already collected are always skipped. These two flags control the edge cases:
RETRY_NA   = True   # re-try pairs stored as "NA" if a CEO name has since been filled in
RETRY_ZERO = False  # re-search pairs that legitimately returned 0 videos

# Stop the run after this many searches (None = keep going until quota runs out).
MAX_SEARCHES_THIS_RUN = None

# Optional: restrict this run to specific years, e.g. [2020, 2021]. None = all years.
YEARS_FILTER = None

# ── Quota bookkeeping ────────────────────────────────────────────────────────
COST_SEARCH = 100
COST_VIDEOS = 1

print(f"CEOs file  : {CEOS_CSV}")
print(f"Output JSON: {OUTPUT_JSON}")

CEOs file  : data/ceos.csv
Output JSON: data/output/videos_metadata/all_videos.json


## 2. API key pool with automatic rotation

Loads every `YOUTUBE_API_KEY_*` from `.env` and rotates to the next key when one runs dry.
When every key is exhausted, `AllKeysExhausted` is raised — the collection loop catches it,
saves, and stops cleanly.

**Detecting an exhausted key is the subtle part.** YouTube reports a spent daily quota two
different ways:

| HTTP | `reason` | meaning |
|------|----------|---------|
| 403 | `quotaExceeded` / `dailyLimitExceeded` | key is done for the day |
| 429 | `rateLimitExceeded` + *"…per day…"* in the message | **also** done for the day |
| 429 | `rateLimitExceeded`, no "per day" | genuine burst limit — just slow down |

The `reason` alone is ambiguous, so the **message text** decides. A daily-quota error rotates
the key; a burst limit retries on the same key with exponential backoff.

**Quota is per Google Cloud project, not per key.** Four keys created inside one project all
draw on the same 10,000 units/day, and rotating between them buys nothing. Google reveals the
project number in the quota error (`consumer 'project_number:…'`), so the pool records it in
`key_projects.json`. Once known, exhausting one key immediately retires its project-mates
instead of wasting a call on each, and a warning is printed at startup.

In [7]:
load_dotenv(ENV_FILE)

# YouTube signals an exhausted DAILY quota in two different ways:
#   403 quotaExceeded / dailyLimitExceeded
#   429 rateLimitExceeded with "Quota exceeded ... per day" in the message  ← looks transient, is not
# Only a short-term burst limit is genuinely retryable, so the message text decides.
QUOTA_REASONS     = {"quotaExceeded", "dailyLimitExceeded"}
TRANSIENT_REASONS = {"rateLimitExceeded", "userRateLimitExceeded", "backendError", "internalError"}

# Phrases that mean "this key is done for the day" even under a transient-looking reason.
DAILY_QUOTA_MARKERS = ("per day", "perday", "dailylimit", "quota exceeded for quota metric")


class AllKeysExhausted(RuntimeError):
    """Every configured YouTube API key has hit its daily quota."""


def _error_info(err: HttpError) -> tuple[str, str, str]:
    """Return (reason, message, project_number) parsed out of an HttpError."""
    reason = message = project = ""
    try:
        payload = json.loads(err.content.decode("utf-8"))
        error = payload.get("error", {})
        message = error.get("message", "") or ""
        errors = error.get("errors") or []
        if errors and isinstance(errors[0], dict):
            reason = errors[0].get("reason", "") or ""
            message = message or errors[0].get("message", "") or ""
    except Exception:
        details = getattr(err, "error_details", None) or []
        if details and isinstance(details[0], dict):
            reason = details[0].get("reason", "") or ""
            message = details[0].get("message", "") or ""
    if not message:
        message = str(err)

    match = re.search(r"project_number:(\d+)", message)
    if match:
        project = match.group(1)
    return reason, message, project


def _is_daily_quota_error(reason: str, message: str, status) -> bool:
    """True when this error means the current key is out of quota for the day."""
    if reason in QUOTA_REASONS:
        return True
    low = message.lower().replace("-", " ")
    if any(marker in low for marker in DAILY_QUOTA_MARKERS):
        return True                      # 429 rateLimitExceeded that is really a daily cap
    return status == 403 and "quota" in low


class ApiKeyPool:
    """Round-robin over the YOUTUBE_API_KEY_* values, skipping exhausted keys."""

    def __init__(self):
        self.keys = []
        i = 1
        while (key := os.getenv(f"YOUTUBE_API_KEY_{i}")):
            self.keys.append((f"YOUTUBE_API_KEY_{i}", key.strip()))
            i += 1
        if not self.keys:
            raise ValueError("No YOUTUBE_API_KEY_* found — check your .env file")

        self.idx = 0
        self.exhausted = set()
        self.units_used = 0
        self._client = None

        # key name -> Google Cloud project number. A key's project is only revealed in its
        # quota-error message, so we cache what we learn and reuse it on later runs.
        self.projects = {}
        if KEY_PROJECTS_CACHE.exists():
            try:
                self.projects = json.loads(KEY_PROJECTS_CACHE.read_text())
            except Exception:
                self.projects = {}

        shared = {}
        for kname, proj in self.projects.items():
            shared.setdefault(proj, []).append(kname)
        for proj, names in shared.items():
            if len(names) > 1:
                print(f"⚠️  {', '.join(names)} are all in Cloud project {proj} — they SHARE one "
                      f"daily quota. Effective capacity is that of a single key.")

    @property
    def name(self) -> str:
        return self.keys[self.idx][0]

    @property
    def client(self):
        if self._client is None:
            self._client = build("youtube", "v3", developerKey=self.keys[self.idx][1],
                                 cache_discovery=False)
        return self._client

    def _note_project(self, project: str):
        """Record which Cloud project a key belongs to, and persist it for future runs."""
        if not project:
            return
        self.projects[self.name] = project
        try:
            KEY_PROJECTS_CACHE.write_text(json.dumps(self.projects, indent=2))
        except Exception:
            pass

    def mark_exhausted(self, project: str = ""):
        """Retire the current key and move to the next live one."""
        self._note_project(project)
        self.exhausted.add(self.idx)
        print(f"    ⚠️  {self.name} out of daily quota ({len(self.exhausted)}/{len(self.keys)} exhausted)")

        # Keys in an already-exhausted project are dead too — don't waste a call proving it.
        # (Only possible for keys whose project we already know, i.e. from the cache.)
        if project:
            for i, (kname, _) in enumerate(self.keys):
                if i not in self.exhausted and self.projects.get(kname) == project:
                    self.exhausted.add(i)
                    print(f"    ⚠️  {kname} is in the same exhausted project ({project}) — skipping it")

        for step in range(1, len(self.keys) + 1):
            nxt = (self.idx + step) % len(self.keys)
            if nxt not in self.exhausted:
                self.idx = nxt
                self._client = None
                print(f"    🔑 switching to {self.name}")
                return
        raise AllKeysExhausted(
            f"All {len(self.keys)} API key(s) have hit their daily quota. "
            f"Progress is saved — re-run this cell after the quota resets (midnight Pacific)."
        )

    def execute(self, make_request, cost: int, max_transient_retries: int = 4):
        """Run `make_request(youtube_client).execute()`, rotating keys / retrying as needed."""
        attempt = 0
        while True:
            try:
                result = make_request(self.client).execute()
                self.units_used += cost
                return result
            except HttpError as err:
                status = getattr(err.resp, "status", None)
                reason, message, project = _error_info(err)

                if _is_daily_quota_error(reason, message, status):
                    self.mark_exhausted(project)   # raises AllKeysExhausted when nothing is left
                    attempt = 0                    # fresh key, fresh retry budget
                    continue

                if reason in TRANSIENT_REASONS or (status is not None and status >= 500):
                    attempt += 1
                    if attempt > max_transient_retries:
                        raise
                    wait = 2 ** attempt
                    print(f"    ⏳ transient error ({reason or status}) — retrying in {wait}s")
                    time.sleep(wait)
                    continue

                raise


pool = ApiKeyPool()
n_projects = len(set(pool.projects.values())) or None

print(f"Loaded {len(pool.keys)} API key(s): {', '.join(n for n, _ in pool.keys)}")
if n_projects and len(pool.projects) == len(pool.keys):
    capacity = n_projects * 10_000 // (COST_SEARCH + COST_VIDEOS)
    print(f"Distinct Cloud projects: {n_projects} → ~{capacity} CEO/year pairs per day")
else:
    capacity = len(pool.keys) * 10_000 // (COST_SEARCH + COST_VIDEOS)
    print(f"Capacity today: up to ~{capacity} CEO/year pairs — but ONLY if each key is in a "
          f"different Google Cloud project.")
    print("Quota is charged per PROJECT, not per key: 4 keys in one project = 10,000 units total,")
    print("not 40,000. Each key's project is learned from its first quota error and cached in")
    print(f"{KEY_PROJECTS_CACHE}.")

Loaded 7 API key(s): YOUTUBE_API_KEY_1, YOUTUBE_API_KEY_2, YOUTUBE_API_KEY_3, YOUTUBE_API_KEY_4, YOUTUBE_API_KEY_5, YOUTUBE_API_KEY_6, YOUTUBE_API_KEY_7
Distinct Cloud projects: 7 → ~693 CEO/year pairs per day


## 3. Load the CEO list and any progress from previous runs

In [8]:
def pair_key(year, company) -> str:
    """Unique id for a CEO/year pair. (year, company) is unique in ceos.csv."""
    return f"{int(year)}|{company}"


def load_progress() -> dict:
    """Read whatever has already been collected. The .jsonl checkpoint wins over the .json."""
    records = {}
    if OUTPUT_JSON.exists():
        with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
            for rec in json.load(f):
                records[pair_key(rec["year"], rec["company"])] = rec
    if OUTPUT_JSONL.exists():
        with open(OUTPUT_JSONL, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rec = json.loads(line)
                    records[pair_key(rec["year"], rec["company"])] = rec  # later lines win
    return records


def append_checkpoint(record: dict):
    """Flush one finished CEO/year pair to disk immediately (O(1), crash-safe)."""
    with open(OUTPUT_JSONL, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def write_all_videos_json(records: dict):
    """(Re)build the deliverable all_videos.json from the in-memory records."""
    ordered = sorted(records.values(), key=lambda r: (-int(r["year"]), r.get("rank") or 9999))
    tmp = OUTPUT_JSON.with_suffix(".json.tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(ordered, f, ensure_ascii=False, indent=2)
    tmp.replace(OUTPUT_JSON)   # atomic — never leaves a half-written file behind
    return OUTPUT_JSON.stat().st_size


ceos = pd.read_csv(CEOS_CSV)
ceos["CEO"] = ceos["CEO"].astype("string").str.strip()
ceos.loc[ceos["CEO"].isin(["", "nan", "None"]), "CEO"] = pd.NA

records = load_progress()

print(f"CEO/year pairs in ceos.csv : {len(ceos):,}")
print(f"  with a CEO name          : {ceos['CEO'].notna().sum():,}")
print(f"  missing a CEO name (NA)  : {ceos['CEO'].isna().sum():,}")
print(f"Already collected          : {len(records):,}")

CEO/year pairs in ceos.csv : 3,000
  with a CEO name          : 2,997
  missing a CEO name (NA)  : 3
Already collected          : 3,500


## 4. Collect the raw video metadata

Re-run this cell as often as you like. It picks up exactly where it left off.

For each pair it does two calls:
1. `search.list` — up to 50 videos for `"[CEO] interview"` published inside that calendar year;
2. `videos.list` — the full `snippet` + `contentDetails` + `statistics` + `status` +
   `topicDetails` record for those video ids.

The stored `videos` entries carry the **complete** API payload, plus `search_rank`
(relevance position) and `video_id` at the top level for convenience.

In [4]:
def build_record(row, videos=None, n_found=None, note=None) -> dict:
    """Assemble the JSON record for one CEO/year pair."""
    ceo = row["CEO"]
    has_ceo = pd.notna(ceo)
    return {
        "year": int(row["year"]),
        "company": row["company"],
        "rank": int(row["rank"]) if pd.notna(row["rank"]) else None,
        "CEO": ceo if has_ceo else None,
        "CEO_alt_names": row["CEO_alt_names"] if pd.notna(row["CEO_alt_names"]) else None,
        "query": QUERY_TEMPLATE.format(ceo=ceo) if has_ceo else None,
        "published_after": f"{int(row['year'])}-01-01T00:00:00Z" if has_ceo else None,
        "published_before": f"{int(row['year'])}-12-31T23:59:59Z" if has_ceo else None,
        # int when we could search, the string "NA" when the CEO name is missing
        "n_videos_found": n_found if n_found is not None else "NA",
        "collected_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "note": note,
        "videos": videos or [],
    }


def search_ceo_year(ceo: str, year: int) -> list:
    """Search YouTube and return the full metadata of every video returned."""
    search = pool.execute(
        lambda yt: yt.search().list(
            part="snippet",
            q=QUERY_TEMPLATE.format(ceo=ceo),
            type="video",
            maxResults=MAX_RESULTS,
            order="relevance",
            publishedAfter=f"{year}-01-01T00:00:00Z",
            publishedBefore=f"{year}-12-31T23:59:59Z",
        ),
        cost=COST_SEARCH,
    )

    items = search.get("items", [])
    if not items:
        return []

    # Keep the relevance ordering from the search response.
    order = {it["id"]["videoId"]: i for i, it in enumerate(items) if it["id"].get("videoId")}
    search_snippets = {it["id"]["videoId"]: it["snippet"] for it in items if it["id"].get("videoId")}

    details = pool.execute(
        lambda yt: yt.videos().list(
            part="snippet,contentDetails,statistics,status,topicDetails",
            id=",".join(order.keys()),
            maxResults=MAX_RESULTS,
        ),
        cost=COST_VIDEOS,
    )
    detail_by_id = {v["id"]: v for v in details.get("items", [])}

    videos = []
    for vid, rank in order.items():
        full = detail_by_id.get(vid)
        if full is not None:
            entry = dict(full)                      # complete videos.list payload
        else:
            # Video vanished between the two calls (deleted/private) — keep the search data.
            entry = {"id": vid, "snippet": search_snippets[vid], "unavailable_in_videos_list": True}
        entry["video_id"] = vid
        entry["search_rank"] = rank
        videos.append(entry)

    videos.sort(key=lambda v: v["search_rank"])
    return videos


def needs_collection(row) -> bool:
    existing = records.get(pair_key(row["year"], row["company"]))
    if existing is None:
        return True
    if existing["n_videos_found"] == "NA":
        return RETRY_NA and pd.notna(row["CEO"])    # CEO name filled in since last run
    if existing["n_videos_found"] == 0:
        return RETRY_ZERO
    return False                                    # already has its videos — never overwrite


# ── Build the work list ──────────────────────────────────────────────────────
todo = ceos if YEARS_FILTER is None else ceos[ceos["year"].isin(YEARS_FILTER)]
todo = todo.sort_values(["year", "rank"], ascending=[False, True])
todo = todo[todo.apply(needs_collection, axis=1)]

n_searchable = int(todo["CEO"].notna().sum())
print(f"Pairs to process this run : {len(todo):,}  ({n_searchable:,} searches + "
      f"{len(todo) - n_searchable:,} NA rows)")
print(f"Estimated quota needed    : {n_searchable * (COST_SEARCH + COST_VIDEOS):,} units "
      f"(available today: ~{len(pool.keys) * 10_000:,})\n")

# ── Main loop ────────────────────────────────────────────────────────────────
searches_done = 0
new_records = 0
stopped_reason = "finished"

try:
    for _, row in todo.iterrows():
        key = pair_key(row["year"], row["company"])

        if pd.isna(row["CEO"]):
            record = build_record(row, note="CEO name missing from ceos.csv — no search performed")
            records[key] = record
            append_checkpoint(record)
            new_records += 1
            continue

        if MAX_SEARCHES_THIS_RUN is not None and searches_done >= MAX_SEARCHES_THIS_RUN:
            stopped_reason = f"reached MAX_SEARCHES_THIS_RUN ({MAX_SEARCHES_THIS_RUN})"
            break

        try:
            videos = search_ceo_year(row["CEO"], int(row["year"]))
        except AllKeysExhausted as exc:
            stopped_reason = str(exc)
            break

        record = build_record(row, videos=videos, n_found=len(videos))
        records[key] = record
        append_checkpoint(record)          # progress saved after every CEO
        searches_done += 1
        new_records += 1

        print(f"[{searches_done:>4}] {row['year']}  {row['CEO']:<28.28} "
              f"{row['company']:<26.26} → {len(videos):>2} videos")

except KeyboardInterrupt:
    stopped_reason = "interrupted by user"

finally:
    size = write_all_videos_json(records)
    print(f"\n{'─' * 70}")
    print(f"Stopped: {stopped_reason}")
    print(f"New/updated records this run : {new_records:,}  ({searches_done:,} searches)")
    print(f"Quota units used this run    : {pool.units_used:,}")
    print(f"Keys exhausted               : {len(pool.exhausted)}/{len(pool.keys)}")
    print(f"Total records on disk        : {len(records):,}")
    print(f"Wrote {OUTPUT_JSON}  ({size / 1024**2:.1f} MB)")

Pairs to process this run : 700  (565 searches + 135 NA rows)
Estimated quota needed    : 57,065 units (available today: ~70,000)

[   1] 2021  Tom Greco                    Advance Auto Parts         → 50 videos
[   2] 2021  Alan B. Colberg              Assurant                   →  1 videos
[   3] 2021  James T. Morris              Pacific Life               → 50 videos
[   4] 2021  Tim Archer                   Lam Research               → 50 videos
[   5] 2021  Michael F. Mahoney           Boston Scientific          → 50 videos
[   6] 2021  Dexter Goei                  Optimum Communications     → 13 videos
[   7] 2021  David Smith                  Sonic Automotive           → 50 videos
[   8] 2021  Lisa Su                      Advanced Micro Devices     → 50 videos
[   9] 2021  David Burritt                United States Steel        → 35 videos
[  10] 2021  Thomas S. Gayner Co Richard  Markel Group               →  0 videos
[  11] 2021  Andres Gluski                AES              

## 5. Summary statistics — how far along are we?

In [5]:
records = load_progress()
write_all_videos_json(records)

summary = pd.DataFrame([
    {
        "year": r["year"],
        "company": r["company"],
        "rank": r["rank"],
        "CEO": r["CEO"],
        "status": "na" if r["n_videos_found"] == "NA" else ("zero" if r["n_videos_found"] == 0 else "ok"),
        "n_videos": 0 if r["n_videos_found"] == "NA" else r["n_videos_found"],
    }
    for r in records.values()
])

total_pairs = len(ceos)
done = len(summary)
pending = total_pairs - done

print("=" * 68)
print("COLLECTION PROGRESS".center(68))
print("=" * 68)
print(f"CEO/year pairs in ceos.csv     : {total_pairs:>7,}")
print(f"  processed                    : {done:>7,}  ({done / total_pairs:6.1%})")
print(f"  still pending                : {pending:>7,}  ({pending / total_pairs:6.1%})")

if done:
    n_ok   = int((summary["status"] == "ok").sum())
    n_zero = int((summary["status"] == "zero").sum())
    n_na   = int((summary["status"] == "na").sum())
    searchable = n_ok + n_zero

    print()
    print(f"  ├─ searched, videos found    : {n_ok:>7,}")
    print(f"  ├─ searched, 0 videos found  : {n_zero:>7,}")
    print(f"  └─ NA (CEO name missing)     : {n_na:>7,}")

    print()
    print("-" * 68)
    print("VIDEOS")
    print("-" * 68)
    print(f"Total videos collected         : {int(summary['n_videos'].sum()):>7,}")
    if searchable:
        searched = summary[summary["status"] != "na"]
        print(f"Mean videos per searched pair  : {searched['n_videos'].mean():>10.1f}")
        print(f"Median videos per pair         : {searched['n_videos'].median():>10.1f}")
        print(f"Pairs hitting the 50 cap       : {int((searched['n_videos'] >= 50).sum()):>7,} "
              f"({(searched['n_videos'] >= 50).mean():.1%})")

    print()
    print("-" * 68)
    print("BY YEAR")
    print("-" * 68)
    expected = ceos.groupby("year").size().rename("in_csv")
    by_year = (summary.groupby("year")
               .agg(processed=("status", "size"),
                    with_videos=("status", lambda s: (s == "ok").sum()),
                    zero=("status", lambda s: (s == "zero").sum()),
                    na=("status", lambda s: (s == "na").sum()),
                    videos=("n_videos", "sum"))
               .join(expected, how="right").fillna(0).astype(int))
    by_year["pending"] = by_year["in_csv"] - by_year["processed"]
    by_year["pct_done"] = (by_year["processed"] / by_year["in_csv"] * 100).round(1)
    print(by_year[["in_csv", "processed", "pending", "with_videos", "zero", "na",
                   "videos", "pct_done"]].sort_index(ascending=False).to_string())

    # ── What's left to do ────────────────────────────────────────────────────
    done_keys = set(records.keys())
    remaining = ceos[~ceos.apply(lambda r: pair_key(r["year"], r["company"]) in done_keys, axis=1)]
    remaining_searches = int(remaining["CEO"].notna().sum())
    n_keys = len(pool.keys) if "pool" in dir() else 4
    per_day = n_keys * 10_000 // (COST_SEARCH + COST_VIDEOS)

    print()
    print("-" * 68)
    print("REMAINING WORK")
    print("-" * 68)
    print(f"Pairs left to process          : {len(remaining):>7,}")
    print(f"  of which need an API search  : {remaining_searches:>7,}")
    print(f"Quota units required           : {remaining_searches * (COST_SEARCH + COST_VIDEOS):>7,}")
    print(f"At {per_day} pairs/day ({n_keys} keys)        : "
          f"{-(-remaining_searches // per_day) if per_day else 0:>7,} more day(s)")

    # ── CEOs with suspiciously little coverage ───────────────────────────────
    zero_hits = summary[summary["status"] == "zero"]
    if len(zero_hits):
        print()
        print(f"Sample of pairs that returned 0 videos ({len(zero_hits):,} total):")
        print(zero_hits[["year", "CEO", "company"]].head(10).to_string(index=False))

                        COLLECTION PROGRESS                         
CEO/year pairs in ceos.csv     :   3,500
  processed                    :   3,500  (100.0%)
  still pending                :       0  (  0.0%)

  ├─ searched, videos found    :   2,926
  ├─ searched, 0 videos found  :     246
  └─ NA (CEO name missing)     :     328

--------------------------------------------------------------------
VIDEOS
--------------------------------------------------------------------
Total videos collected         : 103,821
Mean videos per searched pair  :       32.7
Median videos per pair         :       50.0
Pairs hitting the 50 cap       :   1,742 (54.9%)

--------------------------------------------------------------------
BY YEAR
--------------------------------------------------------------------
      in_csv  processed  pending  with_videos  zero  na  videos  pct_done
year                                                                     
2026     500        500        0          459

## Get videos from second and third CEOs (todo)